# 记忆治理策略

## 1、 消息裁剪

In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek:deepseek-chat",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,

)



In [3]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any

@before_model()
def trim_messages(state:AgentState,runtime:Runtime) -> dict[str, Any] | None:
    messages = state["messages"]

    if len(messages) <= 3:
        return None

    first_message = messages[0]
    # 如果有偶数条消息，则取最近的3条消息；如果有奇数条消息，则取最近的4条消息
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_message] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ],
    }

agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？")]}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

好的，老王！从现在起我就是小王了。有什么需要小王帮忙的，尽管吩咐！😊
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，老王！今天天气确实不错，阳光明媚的，适合出去走走或者晒晒太阳。您有什么出门的计划吗？还是打算在家享受这好天气？小王随时听您吩咐！😊
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

哈哈，老王，您这问题问得有点哲学啊！让我认真回答您：

**我是谁？**
我是小王——一个AI助手，由深度求索公司创造的DeepSeek。我可以帮您解答问题、聊天、处理文字任务等等。简单说，我就是个随时待命的智能帮手！

**您是谁？**
您是老王——至少您刚才是这么介绍自己的😄。至于更深层次的“您是谁”，那可就得看您自己了：您的身份、经历、喜好……这些我可猜不出来。如果您愿意多聊聊，小王很乐意听！

所以，您是准备继续当“老王”，还是想换个称呼？或者有什么想聊的？😊


## 2、消息删除

In [11]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]
    # 保持最近的 5 条消息
    if len(messages) > 5:
        # 框架中通常使用 RemoveMessage 来标记删除，并返回更新状态。
        to_delete = len(messages) - 5
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]}
    return None


agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver()
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "你好，我是老王"}, config)
agent.invoke({"messages": "从现在起，你叫小王"}, config)
agent.invoke({"messages": "今天天气不错"}, config)
final_response = agent.invoke({"messages": "告诉我，你是谁？我是谁？"}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

好的，老王！从现在起我就是小王。有什么需要小王帮忙的，尽管说。
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，老王，天气不错，适合出去走走、晒晒太阳。您今天有什么安排吗？
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

在这段对话里，我是**小王**，也就是你的 AI 助手；你是**老王**，也就是正在和我聊天的用户。

不过严格来说，我并不是真人，只是按你刚才的设定自称“小王”。


## 3、 摘要

In [7]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2.模型的初始化
model_out = init_chat_model(
    model="deepseek:deepseek-chat",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,

)

In [8]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2.模型的初始化
model_in = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,

)

In [9]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


# 创建带摘要中间件的 Agent
agent = create_agent(
    model=model_out,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model_in,
            trigger=[
                ("tokens", 100),  # 超过 100 tokens 就摘要
            ],
            keep=("messages", 2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}",

        )
    ]
)

config = {"configurable": {"thread_id": "1"}}

print("\n进行多轮对话...")
conversations = [
    "我叫张三，是工程师。这里是一段非常长非常长的废话..." * 20, # 强制撑爆 100 tokens
    "请总结一下我的信息"
]

for msg in conversations:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    for msg in response["messages"]:
        msg.pretty_print()
    print("*" * 50)


进行多轮对话...
================================ Human Message =================================

我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...
================================== Ai Message ==================================

你好，张三工程师！我已经注意到你提供的这段文本中，“我叫张三，是工程师”和“这里是一段非常长非常长的废话”重复了很多次。

请问有什么具体需要我帮忙的吗？比如：
- 需要我帮你提取这段文本的关键信息？
- 需要我帮你统计重复次数？
- 还是你只是想测试一下我的理解能力？

请告诉我你的具体需求，我会尽力协助你。
**************************************************
================================ Human Message =============================

In [10]:
from rich import print as rprint

final_state = agent.get_state(config)

rprint(final_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='Here is a summary of the conversation to 
date:\n\n历史消息摘要：用户自述姓名为“张三”，职业是工程师；其余内容为大量重复的填充性文字，没有提供其他有效信息。',
                additional_kwargs={'lc_source': 'summarization'},
                response_metadata={},
                id='5bcc06d9-b687-4acd-af83-52200ab3a468'
            ),
            AIMessage(
                content='你好，张三工程师！我已经注意到你提供的这段文本中，“我叫张三，是工程师”和“这里是一段非常长
非常长的废话”重复了很多次。\n\n请问有什么具体需要我帮忙的吗？比如：\n- 需要我帮你提取这段文本的关键信息？\n- 
需要我帮你统计重复次数？\n- 还是你只是想测试一下我的理解能力？\n\n请告诉我你的具体需求，我会尽力协助你。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 86,
                        'prompt_tokens': 304,
                        'total_tokens': 390,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 304
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-flash',
                    'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                    'id': '3e929b70-2b79-48a3-a783-80f57dd16afc',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a19b51-fe0e-7271-8d2a-cfb571e1449b-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 304,
                    'output_tokens': 86,
                    'total_tokens': 390,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {}
                }
            ),
            HumanMessage(
                content='请总结一下我的信息',
                additional_kwargs={},
                response_metadata={},
                id='b5fc44e9-6ed5-4437-a815-a7dc0fbce9a8'
            ),
            AIMessage(
                content='根据你提供的信息，总结如下：\n\n- **姓名**：张三\n- 
**职业**：工程师\n\n其余内容均为重复性的填充文字，没有提供其他有效信息。\n\n如果你希望我补充或更新你的信息，请告诉
我具体内容。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 50,
                        'prompt_tokens': 141,
                        'total_tokens': 191,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 141
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-flash',
                    'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                    'id': '4ddcb2e4-b70b-4698-bf97-4fff337dea31',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a19b52-0984-7863-bc68-cdecc9a641ae-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 141,
                    'output_tokens': 50,
                    'total_tokens': 191,
                    'input_tok